# Adatok felderítése, pipeline kialakítása

1: SZÜKSÉGES CSOMAGOK IMPORTÁLÁSA

In [2]:
import xarray as xr
from pathlib import Path
import numpy as np
import pandas as pd

2: Data exploratory

In [3]:
files = sorted(Path(r"tigge_data").glob("*.grib"))
print(f"- Összesen {len(files)} fájlt tudtunk letölteni")

# kiválasztunk egy mintát
ds = xr.open_dataset(files[0], engine="cfgrib")

print()
print("- Nézzünk bele egy grib fájlba:")
print(ds)

- Összesen 1 fájlt tudtunk letölteni

- Nézzünk bele egy grib fájlba:
<xarray.Dataset> Size: 195kB
Dimensions:            (time: 31, step: 4, values: 97)
Coordinates:
  * time               (time) datetime64[ns] 248B 2006-10-01 ... 2006-10-31
  * step               (step) timedelta64[ns] 32B 06:00:00 ... 1 days 00:00:00
    valid_time         (time, step) datetime64[ns] 992B ...
    latitude           (values) float64 776B ...
    longitude          (values) float64 776B ...
    number             int64 8B ...
    heightAboveGround  float64 8B ...
    surface            float64 8B ...
Dimensions without coordinates: values
Data variables:
    mx2t6              (time, step, values) float32 48kB ...
    mn2t6              (time, step, values) float32 48kB ...
    t2m                (time, step, values) float32 48kB ...
    tp                 (time, step, values) float32 48kB ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  Eur

In [4]:
varosok = {
    "Budapest": (47.4979, 19.0402),
    "Debrecen": (47.5316, 21.6273),
    "Miskolc":  (48.1035, 20.7784),
    "Pécs":     (46.0727, 18.2323),
    "Szeged":   (46.2530, 20.1414),
}

In [5]:
print(ds.latitude.values)

[48.763994   48.763994   48.763994   48.763994   48.763994   48.763994
 48.763994   48.763994   48.763994   48.763994   48.763994   48.763994
 48.3145566  48.3145566  48.3145566  48.3145566  48.3145566  48.3145566
 48.3145566  48.3145566  48.3145566  48.3145566  48.3145566  48.3145566
 47.86511918 47.86511918 47.86511918 47.86511918 47.86511918 47.86511918
 47.86511918 47.86511918 47.86511918 47.86511918 47.86511918 47.86511918
 47.41568175 47.41568175 47.41568175 47.41568175 47.41568175 47.41568175
 47.41568175 47.41568175 47.41568175 47.41568175 47.41568175 47.41568175
 46.96624431 46.96624431 46.96624431 46.96624431 46.96624431 46.96624431
 46.96624431 46.96624431 46.96624431 46.96624431 46.96624431 46.96624431
 46.51680685 46.51680685 46.51680685 46.51680685 46.51680685 46.51680685
 46.51680685 46.51680685 46.51680685 46.51680685 46.51680685 46.51680685
 46.06736938 46.06736938 46.06736938 46.06736938 46.06736938 46.06736938
 46.06736938 46.06736938 46.06736938 46.06736938 46.06736

In [6]:
def nearest_point(ds, cel_latitude, cel_longitude):
    lat = ds.latitude.values
    lon = ds.longitude.values
    distance = (lat - cel_latitude)**2 + (lon - cel_longitude)**2
    idx = np.argmin(distance)
    return ds.isel(values=idx)

In [7]:
print(nearest_point(ds,varosok["Budapest"][0],varosok["Budapest"][1]))

<xarray.Dataset> Size: 3kB
Dimensions:            (time: 31, step: 4)
Coordinates:
  * time               (time) datetime64[ns] 248B 2006-10-01 ... 2006-10-31
  * step               (step) timedelta64[ns] 32B 06:00:00 ... 1 days 00:00:00
    valid_time         (time, step) datetime64[ns] 992B ...
    number             int64 8B ...
    heightAboveGround  float64 8B ...
    latitude           float64 8B 47.42
    longitude          float64 8B 18.75
    surface            float64 8B ...
Data variables:
    mx2t6              (time, step) float32 496B ...
    mn2t6              (time, step) float32 496B ...
    t2m                (time, step) float32 496B ...
    tp                 (time, step) float32 496B ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Ra

In [8]:
print(ds.step.values)

[21600000000000 43200000000000 64800000000000 86400000000000]


3: Egy hónap dataset-jének kialakítása

In [9]:
def napi_aggregalas(pont_ds):
    """pont_ds: egy adott város rácspontjára kivágott Dataset (time, step dimenziókkal)"""
    
    napi_max  = pont_ds["mx2t6"].max(dim="step")
    napi_min  = pont_ds["mn2t6"].min(dim="step")
    napi_kozep_maxmin = (napi_max + napi_min) / 2
    napi_kozep_t2m    = pont_ds["t2m"].mean(dim="step")
    
    # csapadék: kumulatív, a 24h (utolsó) lépés adja a teljes napi összeget
    napi_csapadek_m  = pont_ds["tp"].isel(step=-1)
    napi_csapadek_mm = napi_csapadek_m * 1000
    
    df = pd.DataFrame({
        "datum": pont_ds["time"].values,
        "tmax": napi_max.values,
        "tmin": napi_min.values,
        "tkozep_maxmin": napi_kozep_maxmin.values,
        "tkozep_t2m": napi_kozep_t2m.values,
        "csapadek_mm": napi_csapadek_mm.values,
    })
    return df

In [10]:
budapest_point = nearest_point(ds, varosok["Budapest"][0], varosok["Budapest"][1])
df = napi_aggregalas(budapest_point)

In [12]:
print(df.head(10))

       datum        tmax        tmin  tkozep_maxmin  tkozep_t2m   csapadek_mm
0 2006-10-01  296.509338  284.494263     290.501801  290.405701 -9.024810e-06
1 2006-10-02  296.138947  285.307709     290.723328  290.889587  4.577637e+01
2 2006-10-03  297.855621  289.658508     293.757080  294.179932 -3.931024e-06
3 2006-10-04  294.084991  286.512787     290.298889  288.218964  3.067017e+03
4 2006-10-05  291.883026  284.959686     288.421356  287.318237 -1.902598e-06
5 2006-10-06  290.791290  280.276398     285.533844  284.847534 -1.338483e-08
6 2006-10-07  291.457367  279.635254     285.546326  285.808685 -4.280513e-13
7 2006-10-08  291.422852  282.029755     286.726318  286.262329  5.340576e+01
8 2006-10-09  291.429108  278.670624     285.049866  285.901550 -2.466796e-07
9 2006-10-10  292.332672  278.916870     285.624756  285.218933 -1.340083e-06


Látjuk hogy kicsit furák az adatok. A hőmérséklet Kelvinben van megadva, a csapadéknál pedig apró, numerikus zajokat figyelhetünk meg.
Próbaljuk ezt megoldani!

In [19]:
def napi_aggregalas(pont_ds):

    kelvin = 273.15

    napi_max  = pont_ds["mx2t6"].max(dim="step") - kelvin
    napi_min  = pont_ds["mn2t6"].min(dim="step") - kelvin
    napi_kozep_maxmin = (napi_max + napi_min) / 2
    napi_kozep_t2m    = pont_ds["t2m"].mean(dim="step") - kelvin
    
    # csapadék: kumulatív, a 24h (utolsó) lépés adja a teljes napi összeget
    napi_csapadek_m  = pont_ds["tp"].isel(step=-1)
    napi_csapadek_mm = (napi_csapadek_m * 1000).clip(min=0)
    
    df = pd.DataFrame({
        "datum": pont_ds["time"].values,
        "d_ta_maxmin": napi_kozep_maxmin.values,
        "d_ta_t2m": napi_kozep_t2m.values,
        "d_tx": napi_max.values,
        "d_tn": napi_min.values,
        "csapadek_mm": napi_csapadek_mm.values,
    })
    return df

In [21]:
df2 = napi_aggregalas(budapest_point)
print(df2)

        datum  d_ta_maxmin   d_ta_t2m       d_tx       d_tn  csapadek_mm
0  2006-10-01    17.351807  17.255707  23.359344  11.344269     0.000000
1  2006-10-02    17.573334  17.739594  22.988953  12.157715    45.776367
2  2006-10-03    20.607071  21.029938  24.705627  16.508514     0.000000
3  2006-10-04    17.148895  15.068970  20.934998  13.362793  3067.016602
4  2006-10-05    15.271362  14.168243  18.733032  11.809692     0.000000
5  2006-10-06    12.383850  11.697540  17.641296   7.126404     0.000000
6  2006-10-07    12.396317  12.658691  18.307373   6.485260     0.000000
7  2006-10-08    13.576309  13.112335  18.272858   8.879761    53.405762
8  2006-10-09    11.899872  12.751556  18.279114   5.520630     0.000000
9  2006-10-10    12.474777  12.068939  19.182678   5.766876     0.000000
10 2006-10-11    11.932541  11.035919  18.664978   5.200104     0.000000
11 2006-10-12    12.838684  11.597595  18.474213   7.203156     0.000000
12 2006-10-13    13.527634  14.220306  18.736877   

Na, máris jófelé tartunk. Viszont találtam a met.hu-n egy leírást ami kicsit árnyalja a helyzetünket.
[met.hu](https://met.hu/eghajlat/magyarorszag_eghajlata/eghajlati_adatsorok/Budapest/leirasok/szamitasok/)